<a href="https://colab.research.google.com/github/audomsak-pypy/Project1-Text_Analytics69/blob/email/SC663402_Part3_3_1_3_3_Email.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part 3: Data Collection Robot + External Business Insight
**SC663402 Data Warehouse and Big Data — งานกลุ่ม (25 คะแนน)**

กลุ่ม: `<ใส่ชื่อกลุ่ม>` | สมาชิก: `<ใส่ชื่อ-รหัสนักศึกษา>`

## โจทย์ทางธุรกิจ
ผู้บริหารอยากรู้ว่า **"ตลาดเทคโนโลยีตอนนี้พูดถึงอะไร อะไรกำลังมา อะไรกำลังไป"** เพื่อใช้วางแผนคอนเทนต์และตัดสินใจลงทุนด้านเทคโนโลยี
ข้อมูลภายในองค์กรตอบเรื่องนี้ไม่ได้ จึงต้องใช้ **ข้อมูลภายนอก (External Data)** คือข่าวเทคโนโลยีจากเว็บไซต์ แล้วสร้าง robot เก็บข้อมูลเอง

**คำถามที่ผู้บริหารต้องการคำตอบ**
1. **Q1** ช่วงนี้ตลาดเทคโนโลยีพูดถึงเรื่องอะไรมากที่สุด
2. **Q2** หัวข้อไหนกำลังโตขึ้น หัวข้อไหนกำลังซา (และเชื่อได้แค่ไหน)
3. **Q3** บริษัทไหนเป็นข่าวมากที่สุด ควรจับตาใครเป็นพิเศษ
4. **Q4** ควรวางแผนคอนเทนต์และลงทุนเทคโนโลยีด้านไหนก่อน

## โครงสร้างสมุดงาน (ตามสไลด์)
| ส่วน | เนื้อหา |
|---|---|
| 3.1 | เลือกแหล่งข้อมูลและวิธีเก็บ → Step 1–7 (Article Collection Robot) |
| 3.2 | ตรวจคุณภาพข้อมูล |
| 3.3 | หา Insight (TF-IDF, Topic Modeling, Trend, Brand) |
| 3.4 | Insight Card สำหรับผู้บริหาร |

In [ ]:
# ===== ติดตั้งและ import (รันเซลล์นี้ก่อนเสมอ) =====
!pip -q install pythainlp
!apt-get -qq install -y fonts-thai-tlwg > /dev/null 2>&1

import os, re, time, random, glob, string
from datetime import datetime
from collections import Counter
from urllib.parse import urljoin, urlparse, urlunparse, parse_qsl, urlencode
from urllib import robotparser

import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm
from bs4 import BeautifulSoup, NavigableString, Comment
from tqdm.auto import tqdm
from scipy.stats import norm

from pythainlp.tokenize import word_tokenize
from pythainlp.corpus import thai_stopwords
from pythainlp.util import normalize

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer, ENGLISH_STOP_WORDS
from sklearn.decomposition import NMF, LatentDirichletAllocation

# ฟอนต์ไทยสำหรับกราฟ (ถ้าไม่ตั้ง ตัวอักษรไทยจะกลายเป็นสี่เหลี่ยม)
for f in glob.glob('/usr/share/fonts/truetype/tlwg/*.ttf'):
    fm.fontManager.addfont(f)
available = {f.name for f in fm.fontManager.ttflist}
for name in ['Loma', 'Garuda', 'Waree']:
    if name in available:
        plt.rcParams['font.family'] = name
        break
plt.rcParams['axes.unicode_minus'] = False

pd.set_option('display.max_colwidth', 70)
pd.set_option('display.width', 170)
random.seed(42)
np.random.seed(42)
print('พร้อมใช้งาน | ฟอนต์กราฟ:', plt.rcParams['font.family'])

---
# 3.1 เลือกแหล่งข้อมูลและวิธีเก็บ

โจทย์กำหนดให้เก็บข้อมูลจากเว็บข่าว Tech แล้วต้องเหลือข้อมูลที่ใช้ได้ **มากกว่า 200 รายการ** หลัง cleaning
ก่อนลงมือเขียน robot กลุ่มเราเทียบวิธีเก็บข้อมูลด้วยหลัก **5V ของ Big Data** (Volume, Velocity, Variety, Veracity, Value) ดังนี้

## ตัวอย่างที่ 2–3: ทางเลือกอื่น
| ทางเลือก | Volume | Velocity (ความสด) | Veracity / กฎการใช้ข้อมูล | ข้อสังเกต |
|---|---|---|---|---|
| RSS / Atom feed (`/atom.xml`) | ต่ำ (ได้เฉพาะข่าวล่าสุดไม่กี่สิบชิ้น) | สูง | ดี (เว็บเปิดให้ใช้) | ย้อนหลังไม่ได้ จึงไม่พอสำหรับเกณฑ์ > 200 รายการ |
| ชุดข้อมูลสำเร็จรูป / API | สูง | มักไม่ใช่ข้อมูลล่าสุด | ต้องดูเงื่อนไขแต่ละชุด | เร็ว แต่ไม่ได้ฝึกขั้นตอน Extract เอง และอาจไม่ตรงโจทย์ Tech trend |
| เว็บข่าวอื่น (เช่น ThaiPBS) | ปานกลาง | สูง | ต้องตรวจ robots.txt ใหม่ | ใช้ pipeline เดียวกันได้ แต่เนื้อหาไม่เน้น Tech ล้วน |
| **Blognone (เลือกใช้)** | **สูง (หลายพันหน้ารวมข่าว)** | **สูง (มีข่าวใหม่ทุกวัน)** | **ท้ายเว็บระบุ Creative Commons Attribution 3.0** | เนื้อหา Tech ภาษาไทยเป็นหลัก เหมาะกับ PyThaiNLP |

> หมายเหตุ: ตารางนี้คือการเปรียบเทียบของกลุ่ม ให้ตรวจว่าสอดคล้องกับ "ตัวอย่างที่ 2–3" ในสไลด์อาจารย์ก่อนส่งงาน และตรวจเงื่อนไขลิขสิทธิ์ท้ายเว็บอีกครั้ง พร้อมอ้างอิง Blognone ในรายงาน

## ตัวอย่างที่ 4: Article Collection Robot — อธิบายทีละขั้น
เราเลือกวิธีนี้ เพราะเป็นวิธีเดียวที่ได้ข่าว **ย้อนหลังหลายร้อยชิ้น** โดยไม่ต้องพึ่ง API
หลักคิดคือ Extract (Step 1–5) → ตรวจ grain (Step 6) → Load ลง corpus (Step 7) ซึ่งตรงกับกระบวนการ **ETL** ของคลังข้อมูลโดยตรง

## Step 1 — กำหนดเว็บไซต์และสิ่งที่ต้องการเก็บ
หลักการของ Data Warehouse คือ **ประกาศ grain ก่อนเก็บข้อมูล** (Declare the grain) เพื่อไม่ให้ตารางปนกันระหว่างระดับข้อมูล
โปรเจกต์นี้กำหนดว่า **1 แถว = 1 ข่าว (1 บทความ)** และใช้เลขท้าย URL (`/node/xxxxx`) เป็นคีย์

In [ ]:
CONFIG = {
    'source_name'      : 'Blognone',
    'base_url'         : 'https://www.blognone.com',
    'listing_url'      : 'https://www.blognone.com/node?page={page}',   # หน้ารวมข่าว (page เริ่มที่ 0)
    # user agent ต้องเป็นตัวอักษร ASCII เท่านั้น (ถ้ามีภาษาไทย requests จะ error) -> แก้อีเมลเป็นของกลุ่มเอง
    'user_agent'       : 'SC663402-StudentProject/1.0 (university coursework; contact: your-email@example.com)',
    'delay_range'      : (1.0, 2.0),   # หน่วงเวลาระหว่างคำขอ (วินาที) เพื่อไม่ให้รบกวนเซิร์ฟเวอร์
    'timeout'          : 20,
    'target_articles'  : 420,          # เก็บดิบให้เกิน 200 เผื่อโดนคัดออกตอน cleaning
    'max_listing_pages': 40,
    'min_valid_records': 200,          # เกณฑ์ของโจทย์: หลัง cleaning ต้องเหลือมากกว่า 200
    'out_dir'          : '/content/data',
}
os.makedirs(CONFIG['out_dir'], exist_ok=True)

schema = pd.DataFrame([
    ('article_id', 'รหัสข่าว (เลขท้าย URL /node/xxxx)',       'หน้ารวมข่าว',  'คีย์ของตาราง: 1 แถวต่อ 1 ข่าว'),
    ('url',        'ลิงก์บทความ (ทำความสะอาดแล้ว)',           'หน้ารวมข่าว',  'อ้างอิงแหล่งที่มา และใช้ตรวจข้อมูลซ้ำ'),
    ('title',      'หัวข้อข่าว',                              'หน้ารวมข่าว',  'ข้อความหลักสำหรับ TF-IDF'),
    ('published',  'วัน-เวลาที่เผยแพร่',                      'หน้ารวมข่าว',  'วิเคราะห์เทรนด์ตามช่วงเวลา'),
    ('author',     'ผู้เขียน (sponsored = เนื้อหาโฆษณา)',     'หน้ารวมข่าว',  'คัดเนื้อหาที่ไม่ใช่ข่าวบรรณาธิการออก'),
    ('tags',       'แท็กที่กองบรรณาธิการติดให้',              'หน้ารวมข่าว',  'ใช้ตรวจความน่าเชื่อถือของหัวข้อที่โมเดลหาได้'),
    ('content',    'เนื้อหาบทความเต็ม',                       'หน้าบทความ',   'วัตถุดิบหลักของ Text Analytics'),
    ('scraped_at', 'เวลาที่ robot เก็บข้อมูล',                'สร้างเอง',     'data lineage: ตรวจย้อนหลังได้ว่าเก็บเมื่อไร'),
], columns=['field', 'ความหมาย', 'ได้จาก', 'ใช้ทำอะไร'])
display(schema)

print('Grain      : 1 row = 1 ข่าว (คีย์ = article_id)')
print('แหล่งข้อมูล :', CONFIG['source_name'], '-', CONFIG['base_url'])
print('เกณฑ์โจทย์  : เหลือข้อมูลที่ใช้ได้ >', CONFIG['min_valid_records'], 'รายการหลัง cleaning')

## Step 2 — ทำความสะอาด URL และตรวจ robots.txt
สองเรื่องนี้ต้องทำก่อนส่งคำขอแรก
- **ทำความสะอาด URL** ลิงก์เดียวกันอาจเขียนได้หลายแบบ (`http`/`https`, มี `#comment`, มี `utm_*`) ถ้าไม่ทำให้เป็นรูปแบบเดียว เราจะเก็บข่าวเดียวกันซ้ำ
- **robots.txt** คือข้อตกลงที่เว็บประกาศว่า robot เข้าหน้าไหนได้ ถือเป็นเรื่อง **จริยธรรมและความถูกต้องของข้อมูล (Veracity/Governance)** โค้ดด้านล่างจะหยุดทำงานถ้า robots.txt ไม่อนุญาต และมี `time.sleep` หน่วงเวลาทุกคำขอ

In [ ]:
def clean_url(url, base=CONFIG['base_url']):
    # ทำให้ URL อยู่ในรูปแบบเดียวกัน กันการเก็บซ้ำเพราะเขียนลิงก์ต่างกัน
    url = urljoin(base + '/', url.strip())
    p = urlparse(url)
    netloc = p.netloc.lower()
    if netloc == 'blognone.com':
        netloc = 'www.blognone.com'
    path = re.sub(r'/+$', '', p.path) or '/'
    keep = [(k, v) for k, v in parse_qsl(p.query) if k == 'page']   # เก็บเฉพาะเลขหน้า ตัด utm ฯลฯ
    return urlunparse(('https', netloc, path, '', urlencode(keep), ''))   # ตัด #fragment ทิ้งด้วย

def is_article_url(url):
    p = urlparse(url)
    return p.netloc.endswith('blognone.com') and re.fullmatch(r'/node/\d+', p.path) is not None

def get_article_id(url):
    m = re.search(r'/node/(\d+)$', urlparse(url).path)
    return int(m.group(1)) if m else None

# ลองกับ URL ที่หน้าตาต่างกันแต่เป็นข่าวเดียวกัน
samples = [
    'http://blognone.com/node/151778/',
    '/node/151778#comment-3',
    'https://www.blognone.com/node/151778?utm_source=facebook',
    'https://www.blognone.com/topics/ai',
]
for s in samples:
    u = clean_url(s)
    print(f'{s:<58} -> {u} | เป็นบทความ: {is_article_url(u)}')

In [ ]:
session = requests.Session()
session.headers.update({'User-Agent': CONFIG['user_agent'], 'Accept-Language': 'th,en;q=0.8'})

# อ่าน robots.txt ก่อนเก็บข้อมูลทุกครั้ง
resp = session.get(CONFIG['base_url'] + '/robots.txt', timeout=CONFIG['timeout'])
print('สถานะ robots.txt:', resp.status_code)
if resp.status_code == 200:
    print(resp.text[:1200])
elif resp.status_code == 404:
    print('(ไม่มีไฟล์ robots.txt = เว็บไม่ได้ประกาศข้อจำกัดไว้)')
else:
    raise RuntimeError('เข้าถึง robots.txt ไม่ได้ ควรหยุดและตรวจสอบก่อนเก็บข้อมูล')

rp = robotparser.RobotFileParser()
rp.parse(resp.text.splitlines() if resp.status_code == 200 else [])

def allowed(url):
    return rp.can_fetch(CONFIG['user_agent'], url)

site_delay = rp.crawl_delay(CONFIG['user_agent'])
if site_delay:
    lo, hi = CONFIG['delay_range']
    CONFIG['delay_range'] = (max(lo, site_delay), max(hi, site_delay + 1))
print('Crawl-delay ที่เว็บกำหนด:', site_delay, '| ช่วงหน่วงเวลาที่ใช้:', CONFIG['delay_range'])

tests = [CONFIG['listing_url'].format(page=0), CONFIG['base_url'] + '/node/151778']
for u in tests:
    print('✔ อนุญาต ' if allowed(u) else '✘ ห้ามเก็บ ', u)
assert all(allowed(u) for u in tests), 'robots.txt ไม่อนุญาตหน้าที่ต้องใช้ -> เปลี่ยนแหล่งข้อมูล (เช่น RSS หรือเว็บอื่น)'

In [ ]:
def next_delay():
    return random.uniform(*CONFIG['delay_range'])

def polite_get(url, retries=3):
    # ขอหน้าเว็บอย่างสุภาพ: เช็ค robots ทุกครั้ง, หน่วงเวลา, ลองใหม่เมื่อโดนจำกัดความเร็ว
    if not allowed(url):
        return None, 'blocked_by_robots'
    for attempt in range(1, retries + 1):
        try:
            r = session.get(url, timeout=CONFIG['timeout'])
        except requests.RequestException:
            time.sleep(3 * attempt)
            continue
        if r.status_code == 200:
            time.sleep(next_delay())
            return r, 'ok'
        if r.status_code in (429, 503):     # เว็บขอให้ช้าลง -> รอนานขึ้นแล้วลองใหม่
            time.sleep(10 * attempt)
            continue
        return None, f'http_{r.status_code}'
    return None, 'request_error'

print('ฟังก์ชัน polite_get พร้อมใช้ (ตรวจ robots + หน่วงเวลา + retry)')

## Step 3 — เข้า "หน้ารวมข่าว" เพื่อหาเฉพาะ Article URLs
หน้ารวมข่าวมีลิงก์ปนกันหลายชนิด (เมนู, แท็ก, โฆษณา, ฟอรัม) เราต้อง **กรองเอาเฉพาะลิงก์บทความ** ซึ่งมีรูปแบบ `/node/ตัวเลข` และอยู่ในหัวข้อข่าว (`h2`)
จากหน้ารวมข่าวเรายังได้ **วันที่ ผู้เขียน และแท็ก** มาพร้อมกัน จึงไม่ต้องเดาจากหน้าบทความ

In [ ]:
META_RE = re.compile(r'By\s+(?P<author>.+?)\s+on\s+(?P<date>\d{1,2}\s+[A-Za-z]+\s+\d{4})\s*-\s*(?P<time>\d{1,2}:\d{2})')

def parse_listing(html_bytes, page_url):
    # อ่านหน้ารวมข่าว 1 หน้า -> รายการข่าว + ลิงก์หน้าถัดไป
    soup = BeautifulSoup(html_bytes, 'html.parser')
    items = []
    for h2 in soup.find_all('h2'):
        a = h2.find('a', href=True)
        if a is None:
            continue
        url = clean_url(a['href'])
        if not is_article_url(url):
            continue
        # เดินไปตาม element ถัดไปจนเจอ h2 ตัวต่อไป หรือปุ่ม Read more = ข้อมูลของข่าวชิ้นนี้
        texts, tags = [], []
        for el in h2.next_elements:
            name = getattr(el, 'name', None)
            if name == 'h2':
                break
            if name == 'a':
                if el.get_text(strip=True).lower() == 'read more':
                    break
                if '/topics/' in el.get('href', ''):
                    tags.append(el.get_text(strip=True))
            elif isinstance(el, NavigableString) and not isinstance(el, Comment):
                texts.append(str(el))
        block = re.sub(r'\s+', ' ', ' '.join(texts))
        m = META_RE.search(block)
        published = None
        if m:
            try:
                published = datetime.strptime(f"{m.group('date')} {m.group('time')}", '%d %B %Y %H:%M')
            except ValueError:
                published = None
        items.append({
            'article_id': get_article_id(url),
            'url': url,
            'title': a.get_text(' ', strip=True),
            'author': m.group('author') if m else None,
            'published': published,
            'tags': ' | '.join(dict.fromkeys(tags)),
        })
    next_a = soup.find('a', rel='next') or soup.find('a', attrs={'title': 'Go to next page'})
    next_url = clean_url(next_a['href']) if next_a is not None and next_a.get('href') else None
    return items, next_url

# ลองกับหน้าแรกก่อน ดูผลด้วยตาว่าดึงถูกไหม
first_url = CONFIG['listing_url'].format(page=0)
r, status = polite_get(first_url)
assert r is not None, f'เปิดหน้ารวมข่าวไม่ได้: {status}'
items, next_url = parse_listing(r.content, first_url)
df_p0 = pd.DataFrame(items)
assert len(df_p0) > 0, 'ไม่พบข่าวเลย -> โครงสร้างหน้าเว็บอาจเปลี่ยน ให้เปิด r.text ดูแล้วปรับ parse_listing'

display(df_p0[['article_id', 'title', 'author', 'published', 'tags']].head(8))
print(f'หน้าแรกพบ {len(df_p0)} ข่าว | อ่านวันที่ไม่ได้ {df_p0["published"].isna().sum()} ข่าว')
print('ลิงก์หน้าถัดไป:', next_url)
print('ข่าวที่ผู้เขียนเป็น sponsored:', (df_p0['author'] == 'sponsored').sum(), '(เนื้อหาโฆษณา จะถูกคัดออกตอน 3.2)')

In [ ]:
def collect_article_urls(max_pages, target):
    rows, seen = [], set()
    page_url = CONFIG['listing_url'].format(page=0)
    empty_streak = 0
    for page in range(max_pages):
        r, status = polite_get(page_url)
        if r is None:
            print(f'หน้า {page}: ดึงไม่ได้ ({status}) -> หยุด')
            break
        items, next_url = parse_listing(r.content, page_url)
        new = [x for x in items if x['url'] not in seen]
        seen.update(x['url'] for x in new)
        rows.extend(new)
        print(f'หน้า {page:>2}: พบ {len(items):>2} ข่าว | ใหม่ {len(new):>2} | สะสม {len(rows)}')
        empty_streak = empty_streak + 1 if not new else 0
        if len(rows) >= target or empty_streak >= 2:
            break
        page_url = next_url or CONFIG['listing_url'].format(page=page + 1)
    return pd.DataFrame(rows)

df_urls = collect_article_urls(CONFIG['max_listing_pages'], CONFIG['target_articles'])
df_urls = df_urls.drop_duplicates('url').reset_index(drop=True)
df_urls.to_csv(f"{CONFIG['out_dir']}/article_urls.csv", index=False, encoding='utf-8-sig')

print('\nรวบรวม Article URLs ได้', len(df_urls), 'ลิงก์')
print('ช่วงวันที่ของข่าว:', df_urls['published'].min(), '->', df_urls['published'].max())
display(df_urls.head())

## Step 4 — เปิดบทความทีละหน้าแล้วดึง "เนื้อหาที่ต้องการ"
หน้าบทความมีของปนหลายอย่าง (เมนู, ความคิดเห็น, ข่าวที่เกี่ยวข้อง, ปุ่มแชร์) ถ้าดึง `get_text()` ทั้งหน้า คำเหล่านี้จะเข้าไปปนใน TF-IDF และทำให้ผลวิเคราะห์เพี้ยน
วิธีที่ใช้: ลองหา container ของเนื้อข่าวตามลำดับ selector แล้ว **ลบส่วนที่เป็น noise** ออก และบันทึกว่าใช้ selector ไหน เพื่อตรวจย้อนหลังได้ (ถ้า Blognone ปรับหน้าเว็บ จะเห็นทันทีจากคอลัมน์ `selector_used`)

In [ ]:
CONTENT_SELECTORS = ['div.node__content', 'div.node-content', 'div.field--name-body',
                     'div.field-name-body', 'article', 'main']
NOISE_RE = re.compile(r'comment|share|social|related|sidebar|advert|banner|breadcrumb|pager|tags|submitted|vote|footer|menu', re.I)

def pick_container(soup):
    for sel in CONTENT_SELECTORS:
        el = soup.select_one(sel)
        if el is not None and len(el.get_text(strip=True)) > 150:
            return el, sel
    # ไม่เจอ selector ไหนเลย -> ใช้ก้อนที่มีย่อหน้า <p> ยาวที่สุดแทน
    best, best_len = None, 0
    for tag in soup.find_all(['article', 'section', 'div']):
        n = sum(len(p.get_text(strip=True)) for p in tag.find_all('p', recursive=False))
        if n > best_len:
            best, best_len = tag, n
    return (best, 'heuristic_p_block') if best_len > 150 else (None, None)

def parse_article(html_bytes):
    soup = BeautifulSoup(html_bytes, 'html.parser')
    for t in soup(['script', 'style', 'noscript', 'nav', 'aside', 'form', 'iframe', 'footer']):
        t.extract()
    container, used = pick_container(soup)
    if container is None:
        return None
    for t in container.find_all(True, class_=NOISE_RE):
        t.extract()
    for t in container.find_all(True, id=NOISE_RE):
        t.extract()
    lines = [ln.strip() for ln in container.get_text('\n').split('\n')]
    text = '\n'.join(ln for ln in lines if ln)
    og = soup.find('meta', attrs={'property': 'og:title'})
    return {'content': text, 'selector_used': used,
            'og_title': og['content'] if og is not None and og.get('content') else None}

# ทดลองกับบทความเดียวก่อน (ไม่เอาข่าว sponsored)
sample_row = df_urls[df_urls['author'] != 'sponsored'].iloc[0]
r, status = polite_get(sample_row['url'])
soup_test = BeautifulSoup(r.content, 'html.parser')

print('ตรวจว่า selector ไหนได้เนื้อหาเท่าไร (ใช้ debug ถ้าเว็บเปลี่ยนโครงสร้าง):')
for sel in CONTENT_SELECTORS:
    el = soup_test.select_one(sel)
    print(f'  {sel:<24}', 'ไม่พบ' if el is None else f'{len(el.get_text(strip=True)):>6} ตัวอักษร')

parsed = parse_article(r.content)
assert parsed is not None, 'ดึงเนื้อหาไม่ได้ -> ดู selector ด้านบนแล้วปรับ CONTENT_SELECTORS'
print('\nหัวข้อจากหน้ารวมข่าว :', sample_row['title'])
print('og:title ของหน้าบทความ:', parsed['og_title'])
print('selector ที่ใช้       :', parsed['selector_used'])
print('ความยาวเนื้อหา        :', len(parsed['content']), 'ตัวอักษร')
print('--- 400 ตัวอักษรแรก ---')
print(parsed['content'][:400])

## Step 5 — ให้ Robot ทำซ้ำกับหลายบทความ
เมื่อฟังก์ชันเก็บ 1 บทความทำงานถูกแล้ว จึงวนซ้ำทั้งรายการ โดยเพิ่มสิ่งที่ robot จริงต้องมี
- **`time.sleep` ทุกคำขอ** (อยู่ใน `polite_get`)
- **try/except และ retry** บทความที่พังไม่ทำให้ทั้งงานล้ม แต่ถูกบันทึกเหตุผลไว้ใน `df_fail`
- **checkpoint** บันทึกความคืบหน้าทุก 25 ข่าว ถ้า Colab หลุดกลางทาง รันซ้ำแล้วจะเก็บต่อจากเดิม

In [ ]:
def scrape_articles(df_urls, checkpoint=None, save_every=25):
    records, failed, done = [], [], set()
    if checkpoint and os.path.exists(checkpoint):
        old = pd.read_csv(checkpoint)
        records = old.to_dict('records')
        done = set(old['url'])
        print(f'พบ checkpoint เดิม {len(done)} ข่าว -> เก็บต่อจากเดิม')
    todo = df_urls[~df_urls['url'].isin(done)]
    for i, row in enumerate(tqdm(todo.to_dict('records'), desc='เก็บบทความ'), start=1):
        r, status = polite_get(row['url'])
        parsed = parse_article(r.content) if r is not None else None
        if parsed is None:
            failed.append({'url': row['url'], 'reason': status if r is None else 'no_content'})
            continue
        records.append({**row, **parsed, 'scraped_at': datetime.now().isoformat(timespec='seconds')})
        if checkpoint and i % save_every == 0:
            pd.DataFrame(records).to_csv(checkpoint, index=False, encoding='utf-8-sig')
    df = pd.DataFrame(records)
    if checkpoint:
        df.to_csv(checkpoint, index=False, encoding='utf-8-sig')
    return df, pd.DataFrame(failed)

# ลองกับ 5 ข่าวก่อน ค่อยรันเต็ม
df_try, fail_try = scrape_articles(df_urls.head(5))
df_try['content_len'] = df_try['content'].str.len()
display(df_try[['article_id', 'title', 'selector_used', 'content_len']])
print('ทดลอง 5 ข่าว: สำเร็จ', len(df_try), '| ล้มเหลว', len(fail_try))

In [ ]:
est_min = len(df_urls) * (np.mean(CONFIG['delay_range']) + 0.6) / 60
print(f'จะเก็บ {len(df_urls)} บทความ ใช้เวลาประมาณ {est_min:.0f} นาที (ห้ามปิดหน้านี้)')

t0 = time.time()
ckpt = f"{CONFIG['out_dir']}/scrape_checkpoint.csv"
df_raw, df_fail = scrape_articles(df_urls, checkpoint=ckpt)
minutes = (time.time() - t0) / 60

print(f'\nเสร็จแล้ว: สำเร็จ {len(df_raw)} | ล้มเหลว {len(df_fail)} | ใช้เวลา {minutes:.1f} นาที')
print(f'อัตราสำเร็จ: {len(df_raw) / max(len(df_urls), 1) * 100:.1f}%')
if len(df_fail):
    display(df_fail['reason'].value_counts().rename('จำนวน').to_frame())
print('\nselector ที่ robot ใช้จริง:')
display(df_raw['selector_used'].value_counts().rename('จำนวนบทความ').to_frame())

## Step 6 — ตรวจว่า "1 row = 1 ข่าว" จริงหรือไม่
นี่คือการตรวจ **grain** ที่ประกาศไว้ใน Step 1 ถ้า grain เพี้ยน (เช่น ข่าวเดียวมี 2 แถว หรือ 1 แถวมีหลายข่าวปนกัน) ค่าที่นับต่อมาทั้งหมดจะผิด เช่น ส่วนแบ่งหัวข้อ และจำนวนข่าวต่อบริษัท
ตรวจ 4 อย่าง: คีย์ไม่ซ้ำ, URL ไม่ซ้ำ, หัวข้อไม่ซ้ำ, และไม่มีแถวที่เนื้อหายาวผิดปกติจนน่าสงสัยว่ารวมหลายข่าว

In [ ]:
df = df_raw.copy()
df['published'] = pd.to_datetime(df['published'], errors='coerce')
df['scraped_at'] = pd.to_datetime(df['scraped_at'], errors='coerce')
df['content_len'] = df['content'].fillna('').str.len()

q3, iqr = df['content_len'].quantile(0.75), df['content_len'].quantile(0.75) - df['content_len'].quantile(0.25)
outlier_limit = q3 + 3 * iqr
grain_check = pd.DataFrame({
    'รายการที่ตรวจ': ['จำนวนแถวทั้งหมด', 'article_id ซ้ำ', 'URL ซ้ำ', 'หัวข้อข่าวซ้ำ',
                      'URL ที่ไม่ใช่รูปแบบ /node/ตัวเลข', f'เนื้อหายาวผิดปกติ (> {outlier_limit:,.0f} ตัวอักษร)'],
    'ผลตรวจ': [len(df), int(df['article_id'].duplicated().sum()), int(df['url'].duplicated().sum()),
               int(df['title'].duplicated().sum()), int((~df['url'].map(is_article_url)).sum()),
               int((df['content_len'] > outlier_limit).sum())],
})
display(grain_check)

# ถ้ามี article_id/URL ซ้ำ ให้คงไว้ 1 แถวต่อ 1 ข่าว
before = len(df)
df = df.drop_duplicates('article_id').reset_index(drop=True)
n_key_dup = before - len(df)
print(f'ตัดแถวซ้ำตามคีย์: {before - len(df)} แถว | เหลือ {len(df)} แถว')
assert df['article_id'].is_unique and df['url'].is_unique, 'grain ยังไม่ใช่ 1 แถว = 1 ข่าว'

print('\nข่าวที่ยาวที่สุด 3 อันดับ (ตรวจด้วยตาว่าเป็นข่าวเดียวจริงไหม):')
display(df.nlargest(3, 'content_len')[['article_id', 'title', 'content_len']])
print('\nตัวอย่างสุ่ม 3 แถว:')
display(df.sample(3, random_state=1).assign(content=lambda d: d['content'].str[:120] + '...')[['article_id', 'title', 'published', 'content']])

In [ ]:
# ตีความผล Step 6
n_dup_title = int(df['title'].duplicated().sum())
print(f'ตีความ: robot เก็บข่าวซ้ำตามคีย์ {n_key_dup} แถว (ตัดออกแล้ว) ตอนนี้ corpus มี {len(df)} แถว และ article_id/URL ไม่ซ้ำเลย จึงยืนยันได้ว่า 1 แถว = 1 ข่าวตามที่ประกาศ grain ไว้')
if n_dup_title:
    print(f'  แต่พบหัวข้อซ้ำ {n_dup_title} แถว ซึ่งอาจเป็นข่าวเดียวกันที่ลงซ้ำคนละ URL จะจัดการในขั้น 3.2 (ตรวจคุณภาพข้อมูล)')
else:
    print('  หัวข้อข่าวไม่ซ้ำ แปลว่าไม่มีข่าวเดียวกันถูกนับสองครั้ง ส่วนแบ่งหัวข้อที่คำนวณต่อไปจึงไม่ถูกขยายเกินจริง')
print('  ผลต่อธุรกิจ: ถ้า grain ผิด ผู้บริหารจะเห็นว่าหัวข้อหนึ่งฮิตกว่าความจริง ทั้งที่เป็นแค่ข่าวเดียวถูกนับซ้ำ')

## Step 7 — บันทึก corpus แล้วค่อยวิเคราะห์
แยก **การเก็บข้อมูล** ออกจาก **การวิเคราะห์** ตามหลักของคลังข้อมูล
- ไฟล์ดิบ (raw corpus) = ชั้น **Data Lake / Staging** เก็บตามที่ robot ได้มา ไม่แก้ไข
- ไฟล์ที่ผ่านการ clean (3.2) = ชั้นพร้อมวิเคราะห์
จากจุดนี้ไปเราวิเคราะห์จากไฟล์ corpus ไม่ต้อง scrape ซ้ำ ทำให้ผลทำซ้ำได้ (Reproducible) และไม่รบกวนเว็บต้นทางอีก

In [ ]:
raw_path = f"{CONFIG['out_dir']}/blognone_corpus_raw.csv"
df.to_csv(raw_path, index=False, encoding='utf-8-sig')
print('บันทึกแล้ว:', raw_path, f'({os.path.getsize(raw_path) / 1024:,.0f} KB)')

# ถ้าอยากเก็บถาวร ให้ยกเลิกคอมเมนต์บรรทัดล่าง (Colab จะลบไฟล์ใน /content เมื่อ session หมดอายุ)
# from google.colab import drive; drive.mount('/content/drive')
# !cp {raw_path} /content/drive/MyDrive/

# อ่านกลับจากไฟล์ เพื่อยืนยันว่าเริ่มวิเคราะห์จาก corpus ที่บันทึกไว้จริง
corpus = pd.read_csv(raw_path, parse_dates=['published', 'scraped_at'])
print('อ่านกลับมา:', corpus.shape, '| คอลัมน์:', list(corpus.columns))

---
# 3.2 ตรวจคุณภาพข้อมูล
**Garbage in, garbage out** ก่อนหา Insight ต้องแน่ใจว่าข้อมูลน่าเชื่อถือ ใช้ **มิติคุณภาพข้อมูล** 5 ด้านเป็นกรอบ

| มิติ | คำถามที่ถาม | ตรวจด้วย |
|---|---|---|
| Completeness (ครบถ้วน) | มีช่องว่างไหม | นับ missing |
| Uniqueness (ไม่ซ้ำ) | ข่าวเดียวถูกเก็บซ้ำไหม | URL/หัวข้อซ้ำ |
| Validity (ใช้ได้จริง) | เนื้อหายาวพอวิเคราะห์ไหม ใช่ข่าวไหม | ความยาว, sponsored, สัดส่วนภาษาไทย |
| Timeliness (ทันสมัย) | ข้อมูลครอบคลุมช่วงเวลาไหน | ช่วงวันที่ |
| Consistency (สม่ำเสมอ) | วิธีดึงเหมือนกันทุกข่าวไหม | selector ที่ใช้ |

**เกณฑ์ตัดข้อมูล (Cleaning rules)** ตามโจทย์: ข้อมูลซ้ำ, ข้อมูลหาย, เนื้อหาสั้น และเพิ่ม 2 กฎที่พบจากข้อมูลจริงคือ เนื้อหาโฆษณา (sponsored) และเนื้อหาที่ไม่ใช่ภาษาไทยเป็นหลัก

In [ ]:
MIN_LEN = 200    # ข่าวสั้นกว่า 200 ตัวอักษร (ราว 2-3 ประโยค) มีคำน้อยเกินกว่าจะให้ TF-IDF จับหัวข้อได้

# ---- (1) ภาพรวมก่อน cleaning ----
profile = pd.DataFrame({
    'dtype': corpus.dtypes.astype(str),
    'missing': corpus.isna().sum(),
    'missing_%': (corpus.isna().mean() * 100).round(1),
    'unique': corpus.nunique(),
})
print('โปรไฟล์ข้อมูลดิบ:')
display(profile)

c = corpus.copy()
c['content_len'] = c['content'].fillna('').str.len()
c['thai_ratio'] = c['content'].fillna('').map(lambda s: len(re.findall(r'[ก-๙]', s)) / max(len(s), 1))

# ---- (2) กฎ cleaning ทีละข้อ ตัดตามลำดับ แล้วดูว่าเหลือเท่าไร ----
rules = [
    ('ข้อมูลหาย (title/published/content ว่าง)', lambda d: d[['title', 'published', 'content']].isna().any(axis=1)),
    ('URL ซ้ำ',                                   lambda d: d['url'].duplicated()),
    ('หัวข้อข่าวซ้ำ',                             lambda d: d['title'].duplicated()),
    ('ไม่ใช่ข่าวบรรณาธิการ (sponsored/press release)',
        lambda d: (d['author'].fillna('').str.lower() == 'sponsored') | d['tags'].fillna('').str.contains('press release', case=False)),
    (f'เนื้อหาสั้นกว่า {MIN_LEN} ตัวอักษร',        lambda d: d['content_len'] < MIN_LEN),
    ('ไม่ใช่ภาษาไทยเป็นหลัก (อักษรไทย < 20%)',     lambda d: d['thai_ratio'] < 0.20),
    ('วันที่ผิดปกติ (อนาคต / ก่อน Blognone ก่อตั้ง)',
        lambda d: (d['published'] > d['scraped_at'] + pd.Timedelta(days=1)) | (d['published'] < pd.Timestamp('2004-08-27'))),
]

remaining = c.copy()
funnel = [{'ขั้นตอน': 'ข้อมูลดิบที่ robot เก็บได้', 'ตัดออก': 0, 'คงเหลือ': len(remaining)}]
for name, rule in rules:
    drop = rule(remaining)
    remaining = remaining[~drop]
    funnel.append({'ขั้นตอน': name, 'ตัดออก': int(drop.sum()), 'คงเหลือ': len(remaining)})
df_clean = remaining.reset_index(drop=True)

funnel = pd.DataFrame(funnel)
print('\nสรุปการทำ cleaning (funnel):')
display(funnel)

n_raw, n_ok = len(c), len(df_clean)
status_txt = '✔ ผ่านเกณฑ์' if n_ok > CONFIG['min_valid_records'] else '✘ ยังไม่ผ่านเกณฑ์ -> เพิ่ม target_articles แล้วเก็บเพิ่ม'
print(f"\nข้อมูลใช้ได้ {n_ok} จาก {n_raw} รายการ ({n_ok / n_raw * 100:.1f}%) | เกณฑ์ > {CONFIG['min_valid_records']} รายการ: {status_txt}")

clean_path = f"{CONFIG['out_dir']}/blognone_corpus_clean.csv"
df_clean.to_csv(clean_path, index=False, encoding='utf-8-sig')
print('บันทึก corpus ที่สะอาดแล้ว:', clean_path)

In [ ]:
# ---- (3) รายงานคุณภาพตาม 5 มิติ (หลัง cleaning) ----
core = ['title', 'published', 'content', 'url']
days_covered = (df_clean['published'].max() - df_clean['published'].min()).days + 1
lag_days = (df_clean['scraped_at'].max() - df_clean['published'].max()).total_seconds() / 86400
primary_sel = df_clean['selector_used'].value_counts(normalize=True).iloc[0] * 100

dq = pd.DataFrame([
    ('Completeness', 'ช่องข้อมูลหลัก 4 ช่องไม่ว่าง (%)',         f"{(1 - df_clean[core].isna().mean().mean()) * 100:.1f}%"),
    ('Uniqueness',   'URL ไม่ซ้ำ (%) / หัวข้อไม่ซ้ำ (%)',         f"{df_clean['url'].nunique() / len(df_clean) * 100:.1f}% / {df_clean['title'].nunique() / len(df_clean) * 100:.1f}%"),
    ('Validity',     f'เนื้อหา ≥ {MIN_LEN} ตัวอักษร (%) / ความยาวมัธยฐาน', f"{(df_clean['content_len'] >= MIN_LEN).mean() * 100:.1f}% / {df_clean['content_len'].median():,.0f} ตัวอักษร"),
    ('Timeliness',   'ช่วงเวลาที่ครอบคลุม / ข่าวล่าสุดเก่ากว่าเวลาเก็บ', f"{days_covered} วัน / {lag_days:.1f} วัน"),
    ('Consistency',  'สัดส่วนบทความที่ใช้ selector หลักตัวเดียวกัน',   f"{primary_sel:.1f}%"),
], columns=['มิติ', 'ตัวชี้วัด', 'ผลลัพธ์'])
display(dq)

# ---- (4) กราฟ ----
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
lim = df_clean['content_len'].quantile(0.99)
ax[0].hist(df_clean['content_len'].clip(upper=lim), bins=30, color='#4C78A8')
ax[0].axvline(df_clean['content_len'].median(), color='red', ls='--', label=f"มัธยฐาน {df_clean['content_len'].median():,.0f}")
ax[0].set_title('การกระจายความยาวเนื้อหาข่าว (ตัวอักษร)')
ax[0].set_xlabel('จำนวนตัวอักษร')
ax[0].legend()

per_day = df_clean.groupby(df_clean['published'].dt.date).size()
ax[1].bar(pd.to_datetime(per_day.index), per_day.values, color='#F58518')
ax[1].set_title('จำนวนข่าวต่อวันใน corpus')
ax[1].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# ---- (5) ตีความผลตรวจคุณภาพ ----
biggest = funnel.iloc[1:].sort_values('ตัดออก', ascending=False).iloc[0]
per_day = df_clean.groupby(df_clean['published'].dt.date).size()
print('ตีความ')
print(f"1) ข้อมูลผ่านเกณฑ์ {n_ok} จาก {n_raw} รายการ สาเหตุที่ตัดมากที่สุดคือ \"{biggest['ขั้นตอน']}\" ({biggest['ตัดออก']} รายการ)")
sp = funnel[funnel['ขั้นตอน'].str.contains('sponsored')]['ตัดออก'].iloc[0]
if sp:
    print(f"   การตัดเนื้อหาโฆษณา {sp} ข่าวสำคัญต่อธุรกิจ: ถ้าเก็บไว้ หัวข้อที่ผู้สนับสนุนจ่ายเงินโปรโมตจะถูกอ่านผิดว่าเป็นเทรนด์ของตลาด")
print(f"2) Volume/Velocity: corpus ครอบคลุม {days_covered} วัน เฉลี่ย {per_day.mean():.1f} ข่าวต่อวัน (ต่ำสุด {per_day.min()} สูงสุด {per_day.max()})")
print(f"3) Validity: ความยาวมัธยฐาน {df_clean['content_len'].median():,.0f} ตัวอักษร เพียงพอให้ TF-IDF แยกหัวข้อได้ เพราะข่าวที่สั้นมากถูกตัดแล้ว")
print(f"4) Consistency: {primary_sel:.0f}% ของบทความดึงด้วย selector ตัวเดียวกัน ถ้าต่ำกว่า 90% แปลว่าโครงสร้างหน้าเว็บไม่สม่ำเสมอ ควรตรวจเนื้อหาด้วยตาก่อนเชื่อผล")
print(f"5) Timeliness: ข้อมูลมีช่วงเวลา {days_covered} วันเท่านั้น จึงบอกได้แค่ 'เทรนด์ระยะสั้น' ไม่ใช่แนวโน้มระยะยาว (บันทึกไว้เป็นข้อจำกัดใน Insight Card)")

---
# 3.3 หา Insight
เปลี่ยนข้อความ (Unstructured Data) ให้เป็นตัวเลข (Structured) แล้วตอบคำถามผู้บริหาร Q1–Q4

| ขั้นตอน | เทคนิค | ตอบคำถาม |
|---|---|---|
| A | ตัดคำไทย (PyThaiNLP) + ทำความสะอาดข้อความ | เตรียมข้อมูล |
| B | TF-IDF หาคำสำคัญ | Q1 |
| C–D | Topic Modeling: NMF (หลัก) และ LDA (ตรวจสอบซ้ำ) | Q1 |
| E | เทรนด์รายสัปดาห์ + ทดสอบทางสถิติ | Q2 |
| F | การกล่าวถึงบริษัท (Brand mention) | Q3 |
| G | ตรวจหัวข้อกับแท็กของบรรณาธิการ (Validation) | ความน่าเชื่อถือ |
| H | สรุปตอบผู้บริหาร | Q1–Q4 |

## A) เตรียมข้อความและตัดคำ
ภาษาไทยไม่มีช่องว่างระหว่างคำ จึงต้องใช้ตัวตัดคำ (`newmm` ของ PyThaiNLP) ก่อนทำ TF-IDF
ข้อควรระวังของข่าวเทคโนโลยี: บริษัทเดียวกันถูกเขียนหลายแบบ (Google / กูเกิล) ถ้าไม่รวมให้เป็นคำเดียว ความถี่จะแตกเป็นหลายคำและถูกประเมินต่ำกว่าจริง จึงทำ **alias mapping** ก่อนตัดคำ

In [ ]:
ALIASES = {   # รวมคำที่หมายถึงสิ่งเดียวกัน (ไทย/อังกฤษสะกดต่างกัน) ให้เป็นคำเดียว
    'artificial intelligence': 'ai', 'ปัญญาประดิษฐ์': 'ai', 'เอไอเอส': 'ais', 'เอไอ': 'ai',
    'กูเกิล': 'google', 'ไมโครซอฟท์': 'microsoft', 'แอปเปิล': 'apple', 'แอปเปิ้ล': 'apple',
    'เอ็นวิเดีย': 'nvidia', 'ซัมซุง': 'samsung', 'โอเพนเอไอ': 'openai',
    'แอนโทรปิก': 'anthropic', 'แอนโธรปิก': 'anthropic', 'ควอลคอมม์': 'qualcomm',
    'เทสลา': 'tesla', 'หัวเว่ย': 'huawei', 'เสียวหมี่': 'xiaomi', 'เสี่ยวหมี่': 'xiaomi',
    'อเมซอน': 'amazon', 'ไอโฟน': 'iphone', 'แอนดรอยด์': 'android', 'ลินุกซ์': 'linux',
    'เอเจนต์': 'agent', 'ชาตจีพีที': 'chatgpt', 'แชตจีพีที': 'chatgpt',
}
ALIAS_RE = re.compile('|'.join(re.escape(k) for k in sorted(ALIASES, key=len, reverse=True)))
URL_RE = re.compile(r'https?://\S+|www\.\S+')
SRC_LINE_RE = re.compile(r'^\s*ที่มา.*$', re.M)      # บรรทัด "ที่มา - ..." เป็นลิงก์อ้างอิง ไม่ใช่เนื้อข่าว

CUSTOM_STOP = set("""ที่มา ข่าว blognone ระบุ กล่าว บอก เปิดเผย ทั้งนี้ อย่างไรก็ตาม ล่าสุด ตอนนี้ ปัจจุบัน
ดังนี้ เพิ่มเติม รายละเอียด นอกจากนี้ สามารถ ต้องการ ทำให้ รวมถึง com www http https""".split())
STOP = set(thai_stopwords()) | set(ENGLISH_STOP_WORDS) | CUSTOM_STOP

def prepare_text(title, content):
    text = f'{title}. {content}'
    text = SRC_LINE_RE.sub(' ', text)
    text = URL_RE.sub(' ', text)
    text = normalize(text.lower())
    return ALIAS_RE.sub(lambda m: f' {ALIASES[m.group(0)]} ', text)

def tokenize(text):
    out = []
    for t in word_tokenize(text, engine='newmm', keep_whitespace=False):
        t = t.strip(string.punctuation + '“”‘’ ')
        if len(t) < 2 or t in STOP:
            continue
        if not re.search(r'[ก-๙a-z]', t):    # ตัดตัวเลขล้วนและสัญลักษณ์
            continue
        out.append(t)
    return out

df_clean['text_nlp'] = [prepare_text(t, c_) for t, c_ in zip(df_clean['title'], df_clean['content'])]
df_clean['tokens'] = [tokenize(t) for t in tqdm(df_clean['text_nlp'], desc='ตัดคำ')]
df_clean['n_tokens'] = df_clean['tokens'].map(len)

i = 0
print('ตัวอย่างข่าว :', df_clean.loc[i, 'title'])
print('หลังตัดคำ    :', df_clean.loc[i, 'tokens'][:25])
print(f"\nคำต่อข่าวเฉลี่ย {df_clean['n_tokens'].mean():.0f} คำ | คำไม่ซ้ำทั้ง corpus {len({t for ts in df_clean['tokens'] for t in ts}):,} คำ")

## B) TF-IDF: หาคำสำคัญ
**หลักการอ่านผล** TF-IDF ให้คะแนนสูงกับคำที่ *ปรากฏบ่อยในข่าวหนึ่ง แต่ไม่ปรากฏทุกข่าว* คำที่ปรากฏเกือบทุกข่าว (เช่น "ผู้ใช้") จะได้คะแนนต่ำโดยอัตโนมัติ
- `min_df=3` ตัดคำที่โผล่แค่ 1–2 ข่าว (อาจเป็นชื่อเฉพาะที่พิมพ์ผิดหรือไม่ใช่เทรนด์)
- `max_df=0.5` ตัดคำที่อยู่เกินครึ่งของข่าว เพราะไม่ช่วยแยกหัวข้อ
- `sublinear_tf=True` ลดอิทธิพลของบทความที่พูดซ้ำคำเดิมหลายรอบ

In [ ]:
MIN_DF, MAX_DF = 3, 0.5
tfidf = TfidfVectorizer(analyzer=lambda toks: toks, min_df=MIN_DF, max_df=MAX_DF, sublinear_tf=True)
X = tfidf.fit_transform(df_clean['tokens'])
terms = np.array(tfidf.get_feature_names_out())
print('ขนาดเมทริกซ์ TF-IDF (ข่าว x คำ):', X.shape)

kw = pd.DataFrame({
    'keyword': terms,
    'mean_tfidf': np.asarray(X.mean(axis=0)).ravel(),
    'docs': np.asarray((X > 0).sum(axis=0)).ravel(),
})
kw['docs_%'] = (kw['docs'] / X.shape[0] * 100).round(1)
top_kw = kw.sort_values('mean_tfidf', ascending=False).head(20).reset_index(drop=True)
display(top_kw)

plt.figure(figsize=(9, 6))
plt.barh(top_kw['keyword'][::-1], top_kw['mean_tfidf'][::-1], color='#4C78A8')
plt.title('20 คำสำคัญสูงสุดจาก TF-IDF (ค่าเฉลี่ยทุกข่าว)')
plt.xlabel('ค่า TF-IDF เฉลี่ย')
plt.tight_layout()
plt.show()

print('ตีความ')
print(f"คำเด่นอันดับ 1-5 คือ: {', '.join(top_kw['keyword'][:5])}")
print(f"คำอันดับ 1 คือ \"{top_kw.loc[0, 'keyword']}\" ปรากฏใน {top_kw.loc[0, 'docs_%']}% ของข่าว ซึ่งบอกว่าเป็นประเด็นที่กว้างที่สุดของช่วงเวลานี้")
print('ถ้าคำสำคัญมีคำทั่วไปที่ไม่ให้ความหมาย (เช่น คำกริยาเชื่อม) ให้เพิ่มลง CUSTOM_STOP แล้วรันเซลล์ A ใหม่ นี่คือวงจรปรับปรุงปกติของงาน Text Analytics')

## C) Topic Modeling ด้วย NMF
TF-IDF บอกได้แค่ "คำไหนสำคัญ" แต่ผู้บริหารอยากรู้ "**เรื่อง**อะไร" จึงใช้ Topic Modeling จัดกลุ่มคำที่มักมาด้วยกันให้เป็นหัวข้อ
**NMF** แยกเมทริกซ์ TF-IDF เป็น *ข่าว × หัวข้อ* และ *หัวข้อ × คำ* (ค่าไม่ติดลบทั้งหมด จึงอ่านเป็น "น้ำหนักของหัวข้อ" ได้ตรง ๆ)

**เลือกจำนวนหัวข้อ (K)** ลอง K ตั้งแต่ 3 ถึง 10 แล้วดู `top10_unique_ratio` = สัดส่วนคำเด่น 10 คำแรกที่ไม่ซ้ำกันข้ามหัวข้อ ถ้าต่ำ แปลว่าหัวข้อซ้ำซ้อนกันเอง
เกณฑ์ที่กลุ่มใช้: เลือก K ที่มากที่สุดที่หัวข้อยังไม่ซ้ำ (≥ 90%) และไม่เกิน 8 เพราะผู้บริหารอ่านได้ไม่เกินราว 8 หัวข้อ

In [ ]:
def top_words_of(components, n=10):
    return [list(terms[row.argsort()[::-1][:n]]) for row in components]

sel = []
for k in range(3, 11):
    m = NMF(n_components=k, init='nndsvd', random_state=42, max_iter=1500)
    m.fit(X)
    tw = top_words_of(m.components_)
    uniq = len({w for t in tw for w in t}) / (k * 10)
    sel.append({'K': k, 'reconstruction_err': round(m.reconstruction_err_, 3), 'top10_unique_ratio': round(uniq, 2)})
sel = pd.DataFrame(sel)
display(sel)

K_OVERRIDE = None          # ใส่ตัวเลขเองได้ถ้าทีมอ่านผลแล้วเห็นว่าควรเปลี่ยน
ok = sel[(sel['top10_unique_ratio'] >= 0.9) & (sel['K'] <= 8)]
K = K_OVERRIDE or (int(ok['K'].max()) if len(ok) else 6)
print(f'เลือก K = {K} หัวข้อ')

In [ ]:
THEMES = {   # พจนานุกรมธีมธุรกิจ ใช้ช่วย "ตั้งชื่อหัวข้อเบื้องต้น" ให้อัตโนมัติ (ทีมต้องอ่านตรวจแล้วแก้ชื่อได้)
    'AI & โมเดลภาษา': ['ai', 'llm', 'โมเดล', 'gemini', 'claude', 'gpt', 'chatgpt', 'openai', 'anthropic', 'agent',
                      'copilot', 'โทเค็น', 'token', 'generative', 'prompt', 'agentic', 'deepseek', 'llama'],
    'ชิป-คลาวด์-ดาต้าเซ็นเตอร์': ['ชิป', 'gpu', 'nvidia', 'tpu', 'cpu', 'npu', 'ram', 'แรม', 'คลาวด์', 'cloud', 'aws',
                      'azure', 'ศูนย์ข้อมูล', 'datacenter', 'server', 'เซิร์ฟเวอร์', 'snapdragon', 'amd', 'intel',
                      'qualcomm', 'soc', 'ประมวลผล', 'พลังงาน', 'tsmc'],
    'มือถือ-อุปกรณ์': ['iphone', 'android', 'ios', 'มือถือ', 'สมาร์ทโฟน', 'samsung', 'galaxy', 'pixel', 'xiaomi', 'ipad',
                      'watch', 'หูฟัง', 'หน้าจอ', 'กล้อง', 'แบตเตอรี่', 'แล็ปท็อป', 'โน้ตบุ๊ก', 'อุปกรณ์', 'wearable', 'แว่น'],
    'ความปลอดภัย-ความเป็นส่วนตัว': ['ช่องโหว่', 'security', 'แฮก', 'hack', 'ransomware', 'ความปลอดภัย', 'malware', 'cve',
                      'รั่ว', 'privacy', 'ความเป็นส่วนตัว', 'patch', 'cyber', 'exploit', 'vulnerability', 'โจมตี',
                      'แฮกเกอร์', 'ฟิชชิง', 'phishing', 'เข้ารหัส'],
    'ซอฟต์แวร์-นักพัฒนา': ['windows', 'linux', 'github', 'โอเพนซอร์ส', 'opensource', 'rust', 'python', 'โค้ด', 'code',
                      'developer', 'นักพัฒนา', 'api', 'ubuntu', 'macos', 'เบราว์เซอร์', 'browser', 'chrome', 'อัพเดต',
                      'อัปเดต', 'เวอร์ชัน', 'ซอฟต์แวร์', 'แอป', 'programming', 'kubernetes', 'docker', 'javascript'],
    'เกม-สตรีมมิง-บันเทิง': ['เกม', 'game', 'games', 'playstation', 'xbox', 'nintendo', 'steam', 'switch', 'ps5', 'netflix',
                      'สตรีมมิง', 'วิดีโอ', 'เพลง', 'ภาพยนตร์', 'disney', 'spotify'],
    'ธุรกิจ-การเงิน-ดีล': ['ผลประกอบการ', 'รายได้', 'กำไร', 'ขาดทุน', 'ipo', 'ลงทุน', 'ดีล', 'ซื้อกิจการ', 'มูลค่า',
                      'ดอลลาร์', 'หุ้น', 'ราคา', 'ไตรมาส', 'acquisition', 'ธุรกิจ', 'สตาร์ทอัพ', 'startup', 'เงินทุน'],
    'นโยบาย-กฎหมาย': ['กฎหมาย', 'กฎระเบียบ', 'eu', 'รัฐบาล', 'ศาล', 'ฟ้อง', 'ค่าปรับ', 'regulation', 'ภาษี', 'สหรัฐ',
                      'จีน', 'ห้าม', 'กสทช', 'pdpa', 'คดี', 'ลิขสิทธิ์'],
    'โซเชียล-แพลตฟอร์ม-แอป': ['facebook', 'instagram', 'tiktok', 'twitter', 'youtube', 'whatsapp', 'line', 'โซเชียล',
                      'ผู้ใช้', 'ครีเอเตอร์', 'แพลตฟอร์ม', 'สตรีม', 'ไลฟ์', 'แชท', 'messenger'],
}

THEME_ACTIONS = {   # ข้อเสนอแนะเชิงธุรกิจตามธีม (ทีมปรับให้เข้ากับบริบทองค์กรได้)
    'AI & โมเดลภาษา': 'ทำคอนเทนต์และเวิร์กช็อปเรื่องการนำ AI ไปใช้จริงในองค์กร และเริ่ม PoC กับงานที่ซ้ำ ๆ ก่อนลงทุนโครงสร้างพื้นฐานขนาดใหญ่',
    'ชิป-คลาวด์-ดาต้าเซ็นเตอร์': 'ประเมินต้นทุนคลาวด์/GPU และพิจารณาทำสัญญาความจุล่วงหน้า เพราะราคาฮาร์ดแวร์ (เช่น แรม) ผันผวน',
    'มือถือ-อุปกรณ์': 'วางแผนคอนเทนต์ตามรอบเปิดตัวอุปกรณ์ และเตรียมระบบ/แอปให้รองรับอุปกรณ์รุ่นใหม่',
    'ความปลอดภัย-ความเป็นส่วนตัว': 'ตั้งกระบวนการ patch management และสื่อสาร security awareness ให้ทีมอย่างสม่ำเสมอ',
    'ซอฟต์แวร์-นักพัฒนา': 'ทบทวน tech stack และทักษะทีมพัฒนา เช่น การเขียนโค้ดด้วย AI ช่วย และเฟรมเวิร์กใหม่',
    'เกม-สตรีมมิง-บันเทิง': 'ใช้เป็นคอนเทนต์เชิงไลฟ์สไตล์เพื่อดึงผู้ชมกลุ่มกว้าง ไม่ควรเป็นเป้าหมายลงทุนหลักด้านเทคโนโลยีองค์กร',
    'ธุรกิจ-การเงิน-ดีล': 'ติดตามดีลควบรวมและผลประกอบการ เพื่อประเมินคู่แข่งและความเสี่ยงของซัพพลายเออร์',
    'นโยบาย-กฎหมาย': 'ตั้งผู้รับผิดชอบติดตามกฎระเบียบ (เช่น PDPA และกฎ AI) และประเมินผลต่อผลิตภัณฑ์',
    'โซเชียล-แพลตฟอร์ม-แอป': 'ปรับกลยุทธ์คอนเทนต์ตามฟีเจอร์ที่แพลตฟอร์มเปลี่ยน และทดลองรูปแบบใหม่แบบต้นทุนต่ำ',
    'อื่นๆ': 'ให้ทีมอ่านตัวอย่างข่าวของหัวข้อนี้แล้วตั้งชื่อเอง ก่อนตัดสินใจ',
}

nmf = NMF(n_components=K, init='nndsvd', random_state=42, max_iter=1500)
W = nmf.fit_transform(X)              # ข่าว x หัวข้อ
H = nmf.components_                   # หัวข้อ x คำ
row_sum = W.sum(axis=1, keepdims=True)
W_norm = np.divide(W, row_sum, out=np.zeros_like(W), where=row_sum > 0)   # น้ำหนักหัวข้อต่อข่าว รวมเป็น 1
df_clean['topic'] = W.argmax(axis=1)

def guess_theme(topic_id, n=15):
    idx = H[topic_id].argsort()[::-1][:n]
    scores = {}
    for theme, kws in THEMES.items():
        s = 0.0
        for j in idx:
            tok = terms[j]
            if any(tok == kw or (len(kw) >= 4 and kw in tok) for kw in kws):
                s += H[topic_id][j]
        scores[theme] = s
    best = max(scores, key=scores.get)
    return best if scores[best] > 0 else 'อื่นๆ'

rows = []
for k in range(K):
    words = list(terms[H[k].argsort()[::-1][:8]])
    best_doc = int(W[:, k].argmax())
    n_k = int((df_clean['topic'] == k).sum())
    theme = guess_theme(k)
    rows.append({'topic': k, 'label': f'T{k} {theme}', 'theme': theme, 'top_words': ', '.join(words),
                 'n_articles': n_k, 'share_%': round(n_k / len(df_clean) * 100, 1),
                 'example_headline': df_clean.loc[best_doc, 'title'][:75]})
topic_table = pd.DataFrame(rows).sort_values('n_articles', ascending=False).reset_index(drop=True)
display(topic_table[['label', 'top_words', 'n_articles', 'share_%', 'example_headline']])

plt.figure(figsize=(9, 4.5))
plt.barh(topic_table['label'][::-1], topic_table['share_%'][::-1], color='#54A24B')
plt.title('สัดส่วนข่าวแยกตามหัวข้อ (NMF)')
plt.xlabel('% ของข่าวทั้งหมด')
plt.tight_layout()
plt.show()

t1 = topic_table.iloc[0]
print('ตีความ')
print(f"หัวข้อที่ใหญ่ที่สุดคือ {t1['label']} ({t1['share_%']}% ของข่าว) คำเด่นคือ {t1['top_words']}")
print(f"3 หัวข้อแรกรวมกันคิดเป็น {topic_table['share_%'][:3].sum():.1f}% ของข่าว แปลว่าความสนใจของตลาดกระจุกตัวอยู่ไม่กี่เรื่อง")
print('ชื่อธีมมาจากการจับคู่คำเด่นกับพจนานุกรมอัตโนมัติ ให้ทีมอ่านคอลัมน์ top_words และ example_headline เพื่อยืนยันหรือแก้ชื่อก่อนนำเสนอ')

## D) ตรวจสอบซ้ำด้วย LDA
หัวข้อจากโมเดลเดียวอาจเป็นผลของอัลกอริทึม จึงรัน **LDA** (โมเดลความน่าจะเป็น ใช้จำนวนคำดิบแทน TF-IDF) แล้วดูว่าได้หัวข้อคล้ายกันไหม
ใช้ค่า **Jaccard** = สัดส่วนคำเด่นที่ซ้ำกันระหว่างสองหัวข้อ ยิ่งสูงยิ่งแปลว่าหัวข้อนั้น "มีอยู่จริงในข้อมูล" ไม่ใช่ผลของวิธีคำนวณ

In [ ]:
cv = CountVectorizer(analyzer=lambda toks: toks, min_df=MIN_DF, max_df=MAX_DF)
Xc = cv.fit_transform(df_clean['tokens'])
terms_c = np.array(cv.get_feature_names_out())
lda = LatentDirichletAllocation(n_components=K, learning_method='batch', max_iter=30, random_state=42)
lda.fit(Xc)

lda_words = [set(terms_c[row.argsort()[::-1][:10]]) for row in lda.components_]
nmf_words = [set(terms[H[k].argsort()[::-1][:10]]) for k in range(K)]

match = []
for k in range(K):
    scores = [len(nmf_words[k] & lw) / len(nmf_words[k] | lw) for lw in lda_words]
    j = int(np.argmax(scores))
    match.append({'topic_nmf': f'T{k}', 'best_lda_topic': j, 'jaccard': round(scores[j], 2),
                  'shared_words': ', '.join(sorted(nmf_words[k] & lda_words[j]))})
match = pd.DataFrame(match)
display(match)

avg_j = match['jaccard'].mean()
n_agree = int((match['jaccard'] >= 0.25).sum())
print('ตีความ')
print(f'หัวข้อ NMF จับคู่กับ LDA ได้เฉลี่ย Jaccard {avg_j:.2f} | {n_agree} จาก {K} หัวข้อมีคำเด่นซ้ำกันตั้งแต่ ~25% ขึ้นไป')
if avg_j >= 0.3:
    print('=> สองวิธีเห็นตรงกันในระดับดี หัวข้อส่วนใหญ่ใช้รายงานผู้บริหารได้')
else:
    print('=> สองวิธีเห็นตรงกันเพียงบางส่วน ควรรายงานเฉพาะหัวข้อที่ Jaccard สูง และระบุว่าหัวข้ออื่นเป็นข้อสังเกตเบื้องต้น')

## E) เทรนด์ตามเวลา (ตอบ Q2)
นับเฉพาะ "ส่วนแบ่งข่าว" ต่อสัปดาห์ (ไม่ใช่จำนวนข่าวดิบ) เพื่อไม่ให้สัปดาห์ที่ข่าวเยอะกว่าเพียงเพราะตีพิมพ์มากถูกอ่านว่าหัวข้อโต
แล้วแบ่งช่วงเวลาที่มีเป็น **ครึ่งแรก vs ครึ่งหลัง** เทียบส่วนแบ่งของแต่ละหัวข้อ และทดสอบด้วย **two-proportion z-test**

**หลักการอ่านผล** ส่วนต่าง (`delta_pp` หน่วยจุดเปอร์เซ็นต์) เป็นบวกแปลว่าโตขึ้น แต่ต้องดู `p_value` ประกอบ: ถ้า p < 0.05 จึงพูดได้ว่า "ต่างกันจริง" ถ้ามากกว่านั้นเป็นเพียงสัญญาณเบื้องต้น (ข้อมูลช่วงสั้น ความแปรปรวนสูงตามธรรมชาติ)

In [ ]:
df_clean['week'] = df_clean['published'].dt.to_period('W').dt.start_time

wk_n = df_clean.groupby('week').size()
weeks_ok = wk_n[wk_n >= 10].index          # ตัดสัปดาห์ที่มีข่าวน้อยเกินไป (มักเป็นสัปดาห์ขอบของช่วงเก็บข้อมูล)
if len(weeks_ok) < 3:
    weeks_ok = wk_n.index

labels = topic_table.sort_values('topic')['label'].tolist()
W_df = pd.DataFrame(W_norm, columns=labels, index=df_clean.index)
W_df['week'] = df_clean['week']
weekly = W_df.groupby('week').mean().loc[weeks_ok] * 100

ax = weekly.plot(figsize=(11, 5), marker='o')
ax.set_title('ส่วนแบ่งหัวข้อข่าวรายสัปดาห์ (%)')
ax.set_ylabel('% ของน้ำหนักหัวข้อ')
ax.set_xlabel('สัปดาห์ (เริ่มวันจันทร์)')
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.show()
print(f'ใช้ {len(weeks_ok)} สัปดาห์ (ตัดสัปดาห์ที่มีข่าว < 10 ชิ้น) | ข่าวต่อสัปดาห์ต่ำสุด-สูงสุด: {wk_n.loc[weeks_ok].min()}-{wk_n.loc[weeks_ok].max()}')

In [ ]:
mid = df_clean['published'].median()
is_early = df_clean['published'] <= mid
n1, n2 = int(is_early.sum()), int((~is_early).sum())

def two_prop_test(x1, n1, x2, n2):
    p1, p2 = x1 / n1, x2 / n2
    p = (x1 + x2) / (n1 + n2)
    se = np.sqrt(p * (1 - p) * (1 / n1 + 1 / n2))
    if se == 0:
        return 0.0, 1.0
    z = (p2 - p1) / se
    return z, 2 * (1 - norm.cdf(abs(z)))

rows = []
for _, t in topic_table.iterrows():
    k = t['topic']
    x1 = int(((df_clean['topic'] == k) & is_early).sum())
    x2 = int(((df_clean['topic'] == k) & ~is_early).sum())
    z, pv = two_prop_test(x1, n1, x2, n2)
    rows.append({'label': t['label'], 'theme': t['theme'], 'share_all_%': t['share_%'],
                 'share_early_%': round(x1 / n1 * 100, 1), 'share_late_%': round(x2 / n2 * 100, 1),
                 'delta_pp': round((x2 / n2 - x1 / n1) * 100, 1), 'p_value': round(pv, 3)})
growth = pd.DataFrame(rows)
growth['significant'] = growth['p_value'] < 0.05

med_share = growth['share_all_%'].median()
def priority(r):
    big, up = r['share_all_%'] >= med_share, r['delta_pp'] > 0
    if big and up:
        return 'ลงทุน/ทำคอนเทนต์ก่อน'
    if big and not up:
        return 'รักษาระดับ'
    if up:
        return 'จับตา/ทดลอง'
    return 'ลดลำดับความสำคัญ'
growth['priority'] = growth.apply(priority, axis=1)

print(f"แบ่งช่วงที่ {mid:%Y-%m-%d}: ครึ่งแรก {n1} ข่าว | ครึ่งหลัง {n2} ข่าว")
display(growth.sort_values('delta_pp', ascending=False))

plt.figure(figsize=(9, 6))
colors = growth['significant'].map({True: '#E45756', False: '#9D9D9D'})
plt.scatter(growth['share_all_%'], growth['delta_pp'], s=160, c=colors, edgecolor='black')
for _, r in growth.iterrows():
    plt.annotate(r['label'], (r['share_all_%'], r['delta_pp']), textcoords='offset points', xytext=(6, 6), fontsize=9)
plt.axhline(0, color='black', lw=0.8)
plt.axvline(med_share, color='black', lw=0.8, ls='--')
plt.xlabel('ส่วนแบ่งข่าวทั้งหมด (%) = ความใหญ่ของหัวข้อ')
plt.ylabel('การเปลี่ยนแปลงครึ่งหลัง - ครึ่งแรก (จุด %) = การเติบโต')
plt.title('แผนที่ลำดับความสำคัญ (แดง = ต่างจริงที่ p < 0.05, เทา = สัญญาณเบื้องต้น)')
plt.tight_layout()
plt.show()

up = growth.sort_values('delta_pp', ascending=False).iloc[0]
down = growth.sort_values('delta_pp').iloc[0]
def sig_text(r):
    return f"p={r['p_value']:.3f} จึงเชื่อได้ว่าเปลี่ยนจริง" if r['p_value'] < 0.05 else f"p={r['p_value']:.2f} ยังสรุปไม่ได้ว่าต่างจริง ถือเป็นสัญญาณเบื้องต้น"
print('ตีความ')
print(f"โตเร็วสุด: {up['label']} จาก {up['share_early_%']}% เป็น {up['share_late_%']}% ({up['delta_pp']:+.1f} จุด) {sig_text(up)}")
print(f"ซาลงมากสุด: {down['label']} จาก {down['share_early_%']}% เป็น {down['share_late_%']}% ({down['delta_pp']:+.1f} จุด) {sig_text(down)}")
print(f"หัวข้อที่ต่างจริงทางสถิติมี {int(growth['significant'].sum())} จาก {K} หัวข้อ (ทดสอบหลายหัวข้อพร้อมกัน จึงควรอ่านผลอย่างระวัง)")

## F) บริษัทที่ถูกกล่าวถึง (ตอบ Q3)
นับ **จำนวนข่าวที่กล่าวถึงบริษัท** (ไม่ใช่จำนวนครั้งที่ปรากฏ) เพราะข่าวเดียวที่เอ่ยชื่อบริษัท 30 ครั้งไม่ควรมีน้ำหนักเท่ากับ 30 ข่าว
รวมชื่อไทย/อังกฤษด้วย alias mapping จากขั้น A เพื่อให้ "กูเกิล" และ "Google" นับเป็นบริษัทเดียวกัน

In [ ]:
BRANDS = {
    'Google': ['google'], 'Microsoft': ['microsoft'], 'Apple': ['apple'], 'Meta': ['meta'],
    'Amazon/AWS': ['amazon', 'aws'], 'Nvidia': ['nvidia'], 'OpenAI': ['openai'], 'Anthropic': ['anthropic'],
    'Samsung': ['samsung'], 'AMD': ['amd'], 'Intel': ['intel'], 'Qualcomm': ['qualcomm'], 'Tesla': ['tesla'],
    'Huawei': ['huawei'], 'Xiaomi': ['xiaomi'], 'Tencent': ['tencent'], 'Alibaba': ['alibaba'],
    'Sony': ['sony'], 'IBM': ['ibm'], 'Oracle': ['oracle'],
}

def brand_mask(words):
    # lookaround กันเคส meta/metadata, apple/pineapple, intel/intelligence
    pat = '|'.join(rf'(?<![a-z0-9]){w}(?![a-z0-9])' for w in words)
    return df_clean['text_nlp'].str.contains(pat, regex=True)

label_of = dict(zip(topic_table['topic'], topic_table['label']))
rows = []
for brand, words in BRANDS.items():
    mk = brand_mask(words)
    x1, x2 = int((mk & is_early).sum()), int((mk & ~is_early).sum())
    _, pv = two_prop_test(x1, n1, x2, n2)
    main_topic = label_of[df_clean.loc[mk, 'topic'].mode().iloc[0]] if mk.sum() else '-'
    rows.append({'brand': brand, 'articles': int(mk.sum()), 'articles_%': round(mk.mean() * 100, 1),
                 'early_%': round(x1 / n1 * 100, 1), 'late_%': round(x2 / n2 * 100, 1),
                 'delta_pp': round((x2 / n2 - x1 / n1) * 100, 1), 'p_value': round(pv, 3), 'main_topic': main_topic})
brand_df = pd.DataFrame(rows).sort_values('articles', ascending=False).reset_index(drop=True)
display(brand_df)

top10 = brand_df.head(10)
plt.figure(figsize=(9, 5))
plt.barh(top10['brand'][::-1], top10['articles_%'][::-1], color='#B279A2')
plt.title('บริษัทที่ถูกกล่าวถึงมากที่สุด (% ของข่าว)')
plt.xlabel('% ของข่าวทั้งหมดที่เอ่ยถึง')
plt.tight_layout()
plt.show()

b1, b2, b3 = brand_df.iloc[0], brand_df.iloc[1], brand_df.iloc[2]
print('ตีความ')
print(f"{b1['brand']} ถูกกล่าวถึงใน {b1['articles_%']}% ของข่าว รองลงมาคือ {b2['brand']} ({b2['articles_%']}%) และ {b3['brand']} ({b3['articles_%']}%)")
print(f"หัวข้อที่ {b1['brand']} เกี่ยวข้องมากสุดคือ {b1['main_topic']} จึงใช้เลือกว่าจะจับตา/พิจารณาเป็นพันธมิตรหรือคู่แข่งในด้านไหน")
rise = brand_df[brand_df['delta_pp'] > 0].sort_values('delta_pp', ascending=False)
if len(rise):
    r0 = rise.iloc[0]
    print(f"บริษัทที่ข่าวเพิ่มขึ้นมากสุดในครึ่งหลังคือ {r0['brand']} ({r0['delta_pp']:+.1f} จุด, p={r0['p_value']:.2f})")
print('ข้อควรระวัง: จำนวนข่าวสะท้อน "ความสนใจของสื่อ" ไม่ใช่ส่วนแบ่งตลาดหรือคุณภาพของสินค้า')

## G) ตรวจหัวข้อกับแท็กของบรรณาธิการ (Validation)
โมเดลหาหัวข้อเองโดยไม่มีเฉลย เราจึงต้องมีวิธีเช็กว่า "เชื่อได้ไหม" ข่าว Blognone มี **แท็กที่กองบรรณาธิการติดเอง** ซึ่งเป็นเหมือนเฉลยหยาบ ๆ
ถ้าหัวข้อที่โมเดลหาได้ตรงกับแท็กที่คนติด ก็เพิ่มความมั่นใจว่าหัวข้อมีความหมายจริง

In [ ]:
tag_lists = df_clean['tags'].fillna('').map(lambda s: [t.strip() for t in s.split('|') if t.strip()])
tag_count = Counter(t for tags in tag_lists for t in tags)
top_tags = pd.DataFrame(tag_count.most_common(15), columns=['tag', 'articles'])

plt.figure(figsize=(8, 5))
plt.barh(top_tags['tag'][::-1], top_tags['articles'][::-1], color='#72B7B2')
plt.title('แท็กที่บรรณาธิการติดมากที่สุด 15 อันดับ')
plt.tight_layout()
plt.show()

def norm_tag(tag):
    return ALIAS_RE.sub(lambda m: ALIASES[m.group(0)], tag.lower())

rows = []
for _, t in topic_table.iterrows():
    k = t['topic']
    idx = np.where(df_clean['topic'] == k)[0]
    cnt = Counter(tag for i in idx for tag in tag_lists.iloc[i])
    top3 = [tg for tg, _ in cnt.most_common(3)]
    words15 = set(terms[H[k].argsort()[::-1][:15]])
    agree = any(w in words15 for tg in top3 for w in norm_tag(tg).split())
    rows.append({'label': t['label'], 'top_words': t['top_words'], 'top3_editor_tags': ', '.join(top3),
                 'tags_match_topic_words': agree})
tag_check = pd.DataFrame(rows)
display(tag_check)

agree_rate = tag_check['tags_match_topic_words'].mean() * 100
print('ตีความ')
print(f"หัวข้อที่แท็กเด่นของบรรณาธิการมีคำตรงกับคำเด่นของโมเดล: {int(tag_check['tags_match_topic_words'].sum())} จาก {K} หัวข้อ ({agree_rate:.0f}%)")
if agree_rate >= 60:
    print('=> โมเดลกับบรรณาธิการเห็นตรงกันเป็นส่วนใหญ่ เชื่อถือหัวข้อได้ในระดับดี')
else:
    print('=> ความตรงกันยังไม่สูง อาจเพราะแท็กเจาะจงกว่าหัวข้อ (ชื่อผลิตภัณฑ์) ให้อ่านคอลัมน์ top3_editor_tags ประกอบก่อนสรุป')

## H) สรุปตอบผู้บริหาร Q1–Q4
เซลล์นี้ประกอบคำตอบจากตัวเลขที่คำนวณไว้ด้านบนโดยอัตโนมัติ ทุกประโยคอ้างอิงตัวเลขจริงในข้อมูลของกลุ่ม

In [ ]:
top3 = topic_table.head(3)
up = growth.sort_values('delta_pp', ascending=False).iloc[0]
down = growth.sort_values('delta_pp').iloc[0]

print('Q1 ตลาดพูดถึงเรื่องอะไรมากที่สุด')
for _, r in top3.iterrows():
    print(f"   - {r['label']}: {r['share_%']}% ของข่าว | คำเด่น: {r['top_words']}")
print(f"   คำสำคัญรวมจาก TF-IDF: {', '.join(top_kw['keyword'][:6])}")

print('\nQ2 อะไรกำลังโต อะไรกำลังซา')
print(f"   โตสุด: {up['label']} ({up['delta_pp']:+.1f} จุด, p={up['p_value']:.3f}) | ซาสุด: {down['label']} ({down['delta_pp']:+.1f} จุด, p={down['p_value']:.3f})")
print(f"   ความน่าเชื่อถือ: มี {int(growth['significant'].sum())} จาก {K} หัวข้อที่ต่างจริงทางสถิติ ที่เหลือใช้เป็นสัญญาณเบื้องต้น")

print('\nQ3 บริษัทไหนเป็นข่าวมากที่สุด')
for _, r in brand_df.head(3).iterrows():
    print(f"   - {r['brand']}: {r['articles_%']}% ของข่าว ({r['delta_pp']:+.1f} จุดในครึ่งหลัง) เกี่ยวข้องกับ {r['main_topic']}")

print('\nQ4 ควรวางแผนคอนเทนต์และลงทุนด้านไหนก่อน')
plan = growth[['label', 'theme', 'share_all_%', 'delta_pp', 'priority']].copy()
plan['action'] = plan['theme'].map(THEME_ACTIONS)
order = {'ลงทุน/ทำคอนเทนต์ก่อน': 0, 'จับตา/ทดลอง': 1, 'รักษาระดับ': 2, 'ลดลำดับความสำคัญ': 3}
plan = plan.sort_values(['priority', 'share_all_%'], key=lambda s: s.map(order) if s.name == 'priority' else -s).reset_index(drop=True)
display(plan)
for _, r in plan.head(3).iterrows():
    print(f"   [{r['priority']}] {r['label']} -> {r['action']}")

## ต่อยอดเข้า Data Warehouse (Star Schema)
เพื่อให้ผลลัพธ์นำไปใช้ต่อในระบบ BI ได้ เราจัด corpus ที่วิเคราะห์แล้วเป็น **Star Schema** ตามหลักการออกแบบคลังข้อมูล
- **Fact**: `fact_article` (grain = 1 ข่าว)
- **Dimension**: `dim_date`, `dim_topic`
- **Bridge**: `bridge_article_tag` (ข่าว 1 ชิ้นมีหลายแท็ก จึงเป็นความสัมพันธ์แบบ many-to-many)

In [ ]:
dim_date = pd.DataFrame({'date': pd.to_datetime(sorted(df_clean['published'].dt.date.unique()))})
dim_date['date_key'] = dim_date['date'].dt.strftime('%Y%m%d').astype(int)
dim_date['week_start'] = dim_date['date'].dt.to_period('W').dt.start_time
dim_date['month'] = dim_date['date'].dt.strftime('%Y-%m')

dim_topic = topic_table[['topic', 'label', 'theme', 'top_words']].rename(columns={'topic': 'topic_key'})

fact_article = pd.DataFrame({
    'article_id': df_clean['article_id'],
    'date_key': df_clean['published'].dt.strftime('%Y%m%d').astype(int),
    'topic_key': df_clean['topic'],
    'author': df_clean['author'],
    'n_tokens': df_clean['n_tokens'],
    'content_len': df_clean['content_len'],
    'url': df_clean['url'],
})

bridge_article_tag = (pd.DataFrame({'article_id': df_clean['article_id'], 'tag': tag_lists})
                      .explode('tag').dropna().query("tag != ''").reset_index(drop=True))

for name, tbl in [('dim_date', dim_date), ('dim_topic', dim_topic), ('fact_article', fact_article),
                  ('bridge_article_tag', bridge_article_tag)]:
    tbl.to_csv(f"{CONFIG['out_dir']}/{name}.csv", index=False, encoding='utf-8-sig')
    print(f'{name:<20} {tbl.shape[0]:>5} แถว x {tbl.shape[1]} คอลัมน์')
display(fact_article.head())
topic_table.to_csv(f"{CONFIG['out_dir']}/topic_summary.csv", index=False, encoding='utf-8-sig')

---
# 3.4 Insight Card สำหรับผู้บริหาร
การ์ดใบเดียวที่ผู้บริหารอ่านจบใน 1 นาที: **คำถาม → Insight → หลักฐาน → ข้อเสนอแนะ → ข้อจำกัด**
เซลล์ถัดไปสร้างการ์ดจากตัวเลขจริงโดยอัตโนมัติ ทีมควรอ่านทวนและปรับถ้อยคำก่อนส่งงาน

In [ ]:
best_topic = topic_table.iloc[0]
rising = growth.sort_values('delta_pp', ascending=False).iloc[0]
brands3 = brand_df.head(3)
acts = plan[plan['priority'].isin(['ลงทุน/ทำคอนเทนต์ก่อน', 'จับตา/ทดลอง'])].head(3)
if acts.empty:
    acts = plan.head(3)

date_from, date_to = df_clean['published'].min().date(), df_clean['published'].max().date()
sig_word = 'ต่างจริงทางสถิติ' if rising['p_value'] < 0.05 else 'ยังเป็นสัญญาณเบื้องต้น'

lines = []
lines.append('=' * 74)
lines.append('INSIGHT CARD | เทรนด์ข่าวเทคโนโลยี เพื่อวางแผนคอนเทนต์และการลงทุน')
lines.append('=' * 74)
lines.append('1) คำถามธุรกิจ')
lines.append('   ตลาดเทคโนโลยีกำลังพูดถึงอะไร อะไรกำลังมา และเราควรวางแผนคอนเทนต์/ลงทุนด้านไหนก่อน')
lines.append('')
lines.append('2) Insight หลัก')
lines.append(f"   - หัวข้อที่พูดถึงมากที่สุด: {best_topic['label']} ({best_topic['share_%']}% ของข่าว) คำเด่น: {best_topic['top_words']}")
lines.append(f"   - หัวข้อที่โตเร็วสุดในครึ่งหลัง: {rising['label']} ({rising['delta_pp']:+.1f} จุด %, {sig_word})")
lines.append('   - บริษัทที่เป็นข่าวมากสุด: ' + ', '.join(f"{r['brand']} ({r['articles_%']}%)" for _, r in brands3.iterrows()))
lines.append('')
lines.append('3) หลักฐาน')
lines.append(f"   - ข่าวที่ใช้วิเคราะห์ {len(df_clean)} ข่าว จากแหล่ง Blognone ช่วง {date_from} ถึง {date_to} ({days_covered} วัน)")
lines.append(f"   - NMF หา {K} หัวข้อ; ตรวจซ้ำกับ LDA ได้ Jaccard เฉลี่ย {avg_j:.2f}; ตรงกับแท็กบรรณาธิการ {agree_rate:.0f}% ของหัวข้อ")
lines.append(f"   - ข้อมูลผ่านการทำความสะอาด {len(df_clean)}/{n_raw} รายการ ตัดโฆษณา ข่าวซ้ำ ข่าวสั้น และข่าวที่ไม่ใช่ภาษาไทยออกแล้ว")
lines.append('')
lines.append('4) ข้อเสนอแนะ')
for i, (_, r) in enumerate(acts.iterrows(), start=1):
    lines.append(f"   {i}. [{r['priority']}] {r['label']}: {r['action']}")
lines.append('')
lines.append('5) ข้อจำกัดของข้อมูล')
lines.append('   - แหล่งข้อมูลเดียว (Blognone) สะท้อนมุมมองของสื่อไทยด้านเทคโนโลยี ไม่ใช่ทั้งตลาดหรือความต้องการของลูกค้า')
lines.append(f"   - ช่วงเวลาสั้น ({days_covered} วัน) บอกได้แค่เทรนด์ระยะสั้น ไม่ควรอนุมานเป็นแนวโน้มระยะยาว")
lines.append('   - จำนวนข่าวสะท้อนความสนใจของสื่อ ไม่ใช่ยอดขายหรือส่วนแบ่งตลาด และไม่ได้บอกความเป็นเหตุเป็นผล')
lines.append('   - หัวข้อจาก NMF เป็นการจัดกลุ่มทางสถิติ ชื่อธีมต้องให้คนตรวจ และการตัดคำภาษาไทยอาจคลาดเคลื่อนกับชื่อเฉพาะใหม่ ๆ')
lines.append('   - ทดสอบหลายหัวข้อพร้อมกัน ค่า p ควรอ่านอย่างระวัง เพราะมีโอกาสพบความต่างโดยบังเอิญ')
lines.append('')
lines.append('6) ขั้นต่อไป: เพิ่มแหล่งข่าวและช่วงเวลา (เช่น ThaiPBS) แล้วรัน pipeline เดิมซ้ำ เพื่อยืนยันเทรนด์ก่อนตัดสินใจลงทุน')
lines.append('=' * 74)

card = '\n'.join(lines)
print(card)

card_path = f"{CONFIG['out_dir']}/insight_card.md"
with open(card_path, 'w', encoding='utf-8') as fh:
    fh.write(card)
print('\nบันทึกการ์ดแล้ว:', card_path)

---
# สรุปตามหลักรายวิชา และสิ่งที่ทีมต้องทำก่อนส่ง

| แนวคิดในรายวิชา | ที่ใช้ในงานนี้ |
|---|---|
| Big Data 5V | Volume/Velocity (ข่าวรายวัน), Variety (ข้อความไม่มีโครงสร้าง), Veracity (robots.txt, คัด sponsored), Value (Insight Card) |
| ETL | Extract = Step 1–5, Transform = 3.2 และขั้น A, Load = Step 7 และ Star Schema |
| Grain | ประกาศ "1 แถว = 1 ข่าว" ใน Step 1 และตรวจใน Step 6 |
| Data Quality | 5 มิติใน 3.2 พร้อม funnel การตัดข้อมูล |
| Unstructured → Structured | TF-IDF, NMF, LDA แปลงข้อความเป็นเมทริกซ์และหัวข้อ |
| Dimensional Modeling | fact_article, dim_date, dim_topic, bridge_article_tag |

**ก่อนส่งงาน**
1. แก้อีเมลใน `user_agent` (Step 1) เป็นของกลุ่มเอง
2. รันทั้งสมุดงานตั้งแต่ต้นถึงท้าย (Runtime → Run all) แล้วอ่านผลจริง
3. แก้ชื่อธีมของแต่ละหัวข้อให้ตรงกับ `top_words` และ `example_headline` (ทีมต้องอธิบายได้เอง)
4. ทวนข้อความตีความและ Insight Card ให้ตรงกับตัวเลขที่ได้ และตรวจตารางทางเลือก (ตัวอย่างที่ 2–3) ให้ตรงกับสไลด์
5. อ้างอิง Blognone ในรายงาน (ท้ายเว็บระบุ Creative Commons Attribution 3.0 ให้ตรวจเงื่อนไขอีกครั้ง)